In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [ ]:
# General stochastic generation (CoSMoS_py) — discharge, temperature, ...
from pyhydra.climate.stochastic_generation import (
    analyze_ts,
    report_ts,
    simulate_ts,
    generate_ts,
    fit_distribution,
    fit_acs,
)

# Precipitation stochastic generation (NEOPRENE NSRP)
from pyhydra.climate.stochastic_generation import NSRPModel, STNSRPModel

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 🎲 Stochastic Time Series Generation

## 📚 General Description

This notebook demonstrates two complementary approaches to stochastic generation available in `pyhydra`:

| Approach | Model | Variables | Library |
|----------|-------|-----------|----------|
| **General** | Seasonal stochastic (CoSMoS) | Discharge, temperature, any | CoSMoS_py |
| **Precipitation** | NSRPM (Neyman-Scott) | Precipitation only | NEOPRENE |

### 🔧 Installation requirements

```bash
# CoSMoS_py (general stochastic)
pip install -e /path/to/CoSMoS_py

# NEOPRENE (NSRP for precipitation)
pip install NEOPRENE
```

---

## 📦 Output

Both approaches generate synthetic time series that **preserve observed statistics**:
- Marginal distribution (mean, variance, skewness)
- Temporal autocorrelation structure
- Seasonal variability

---
## 1. 🔵 General Stochastic Generation — CoSMoS_py

CoSMoS fits a **seasonal stochastic model** to any hydro-meteorological variable and generates synthetic realisations that match the observed:
- Seasonal mean and standard deviation
- Marginal probability distribution
- Autocorrelation structure (ACS)

### 💡 Typical use case
Generate an ensemble of synthetic discharge series for uncertainty quantification in hydrological impact studies.

In [ ]:
# Generate a synthetic 30-year daily discharge series (observed)
rng   = np.random.default_rng(42)
dates = pd.date_range("1990-01-01", "2019-12-31", freq="D")
n     = len(dates)
doy   = dates.dayofyear

seasonal_mean = 30 + 20 * np.sin(2 * np.pi * (doy / 365 - 0.3))
Q = np.maximum(0.1, seasonal_mean + rng.normal(0, 8, n))
Q_obs = pd.Series(Q, index=dates, name="discharge")

print(f"Period   : {dates[0].date()} → {dates[-1].date()}")
print(f"Mean Q   : {Q_obs.mean():.2f} m³/s")
print(f"Std  Q   : {Q_obs.std():.2f} m³/s")

In [ ]:
# Fit a marginal distribution to the observed series
# (requires CoSMoS_py installed)

# dist = fit_distribution(Q_obs, dist_name="Gamma")
# print("Fitted distribution:", dist)

# Fit the autocorrelation structure
# acs = fit_acs(Q_obs, lags=30)
# print("Fitted ACS:", acs)

print("(Requires CoSMoS_py — pip install -e /path/to/CoSMoS_py)")

In [ ]:
# Analyse the observed series (compute statistics)
# ts_stats = analyze_ts(Q_obs)
# report_ts(ts_stats)
print("(Requires CoSMoS_py)")

In [ ]:
# Generate N synthetic realisations of the same length
# N_SIM = 100

# simulations = simulate_ts(
#     obs=Q_obs,
#     n_simulations=N_SIM,
#     dist_name="Gamma",
# )

# fig, ax = plt.subplots(figsize=(14, 5))
# for sim in simulations[:20]:                      # plot 20 realisations
#     ax.plot(sim.index, sim.values, lw=0.4, alpha=0.3, color="steelblue")
# ax.plot(Q_obs.index, Q_obs.values, lw=1.0, color="black", label="Observed")
# ax.set_ylabel("Discharge (m³/s)")
# ax.set_title(f"Stochastic generation — {N_SIM} realisations", fontsize=13)
# ax.legend()
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()
print("(Simulation commented out — requires CoSMoS_py)")

---
## 2. 🌧️ Precipitation — NSRP Model (NEOPRENE)

The **Neyman-Scott Rectangular Pulses Model (NSRPM)** is a process-based stochastic model
specifically designed for precipitation:

- **Physical basis**: storm arrivals (Poisson) → cell clusters (Poisson) → rectangular pulses (exponential duration/intensity)
- **Calibration**: Particle Swarm Optimisation (PSO) matching observed statistics
- **Target statistics**: mean, variance, covariance, probability dry, skewness — by month

### 📐 Model variants

| Class | Model | Sites |
|-------|-------|-------|
| `NSRPModel` | NSRPM | Single-site |
| `STNSRPModel` | ST-NSRPM | Multi-site (spatial) |

> **Requires NEOPRENE:** `pip install NEOPRENE`  
> Developed at IH Cantabria: https://github.com/IHCantabria/NEOPRENE

In [ ]:
# Generate a synthetic 30-year daily precipitation series
dates_p   = pd.date_range("1990-01-01", "2019-12-31", freq="D")
prob_wet  = 0.30 + 0.15 * np.sin(2 * np.pi * (dates_p.dayofyear / 365 - 0.1))
wet       = rng.random(len(dates_p)) < prob_wet
prec      = np.where(wet, rng.exponential(8, len(dates_p)), 0.0)
P_obs     = pd.Series(prec, index=dates_p, name="precipitation")

print(f"Wet-day fraction : {(P_obs > 0).mean():.2f}")
print(f"Mean             : {P_obs.mean():.2f} mm/day")
print(f"Std              : {P_obs.std():.2f} mm/day")
print(f"Max              : {P_obs.max():.1f} mm")

In [ ]:
# Instantiate the single-site NSRP model
model = NSRPModel(
    statistics=["mean", "variance", "covariance", "probability_dry"],
    weights=None,   # uniform weights — all statistics equally important
)
print(model)

In [ ]:
# Compute observed statistics (always available — no NEOPRENE needed)
obs_stats = model.compute_statistics(P_obs)
print("Observed statistics (monthly):")
obs_stats

In [ ]:
# Calibrate the NSRP model (requires NEOPRENE)
# model.calibrate(P_obs, verbose=True)
# print(model.summary())
print("(Calibration requires NEOPRENE — pip install NEOPRENE)")

In [ ]:
# Simulate 100 years of synthetic precipitation (requires NEOPRENE)
# df_sim = model.simulate(year_ini=1990, year_fin=2090)

# fig, axes = plt.subplots(2, 1, figsize=(14, 7))
# axes[0].bar(P_obs.index, P_obs.values, width=1, color="royalblue", alpha=0.7, label="Observed")
# axes[0].set_ylabel("Precipitation (mm)")
# axes[0].set_title("Observed daily precipitation")
# axes[0].set_xlim(P_obs.index[0], P_obs.index[0] + pd.Timedelta(days=365))
# axes[0].grid(alpha=0.3, axis='y')

# axes[1].bar(df_sim.index, df_sim.values, width=1, color="orange", alpha=0.7, label="Simulated")
# axes[1].set_ylabel("Precipitation (mm)")
# axes[1].set_title("Simulated daily precipitation (1 year sample)")
# axes[1].set_xlim(df_sim.index[0], df_sim.index[0] + pd.Timedelta(days=365))
# axes[1].grid(alpha=0.3, axis='y')

# plt.tight_layout()
# plt.show()
print("(Simulation commented out — requires NEOPRENE)")

---
## 3. 🗺️ Multi-site precipitation — ST-NSRP Model

The **Spatial-Temporal NSRPM (ST-NSRPM)** extends the single-site model to reproduce
**spatial correlation** between multiple rainfall gauges simultaneously.

In [ ]:
# Multi-site model instantiation
st_model = STNSRPModel(
    statistics=["mean", "variance", "covariance", "probability_dry"],
    attributes=["cross_correlation"],    # spatial statistics
)
print(st_model)
print("\nCalibrate with a DataFrame (dates × stations):")
print("  st_model.fit(df_multisite)")
print("  df_sim = st_model.simulate(year_ini=1990, year_fin=2090)")

---
## 4. 📊 Statistics validation

After generating synthetic series, validate that observed statistics are reproduced.

In [ ]:
# Compare observed vs simulated monthly statistics
# (run after calibration and simulation)

# obs_monthly = P_obs.resample('ME').agg(['mean', 'std', lambda x: (x == 0).mean()])
# sim_monthly = df_sim.resample('ME').agg(['mean', 'std', lambda x: (x == 0).mean()])

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, col, title, ylabel in zip(
#     axes,
#     ['mean', 'std', '<lambda_0>'],
#     ['Mean daily precipitation', 'Std daily precipitation', 'Dry-day fraction'],
#     ['mm/day', 'mm/day', 'fraction'],
# ):
#     ax.plot(range(1,13), obs_monthly.groupby(obs_monthly.index.month)[col].mean(), 'ko-', label='Observed')
#     ax.plot(range(1,13), sim_monthly.groupby(sim_monthly.index.month)[col].mean(), 'r^--', label='Simulated')
#     ax.set_xticks(range(1,13))
#     ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
#     ax.set_title(title, fontsize=11)
#     ax.set_ylabel(ylabel)
#     ax.legend(fontsize=9)
#     ax.grid(alpha=0.3)
# plt.suptitle('Observed vs Simulated monthly statistics', fontsize=13)
# plt.tight_layout()
# plt.show()

print("(Validation plot commented out — run after calibration + simulation)")